In [0]:
import dataiku
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def process_postgres_datasets(dry_run=True):
    """
    Iterates through project datasets using object references, 
    identifies 'postgres' types, and updates the table name parameter 
    to '${projectKey}_datasetName'.
    
    Args:
        dry_run (bool): If True, prints changes without applying them. 
                        If False, applies changes to the dataset settings.
    """
    
    client = dataiku.api_client()
    project = client.get_default_project()
    
    # 1. Iterate directly as objects
    # This returns a list of dataiku.Dataset objects
    datasets = project.list_datasets(as_type="objects")
    
    logging.info(f"Starting process in {'DRY RUN' if dry_run else 'LIVE'} mode.")
    logging.info(f"Found {len(datasets)} datasets in project {project.project_key}.")

    for dataset in datasets:
        # We must load the settings to inspect the type and params
        try:
            logging.debug(f"Accessing {dataset.name}...")
            settings = dataset.get_settings()
        except Exception as e:
            logging.error(f"Could not load settings for dataset {dataset.name}: {e}")
            continue

        # 2. Filter for postgres type
        if settings.type == 'postgres':
            # Access the raw params dictionary
            params = settings.get_raw_params()
            
            # Safely get current table name (default to None if missing)
            current_table_name = params.get('table')
            
            # Construct the target name using double curly braces to escape the f-string
            # This ensures ${projectKey} remains a literal variable string
            target_table_name = f"${{projectKey}}_{dataset.name}"

            # 3. Idempotency Check: Only proceed if names are different
            if current_table_name != target_table_name:
                log_msg = (
                    f"Dataset: {dataset.name} | "
                    f"Current Table: {current_table_name} -> "
                    f"New Table: {target_table_name}"
                )
                
                if dry_run:
                    logging.info(f"[WOULD CHANGE] {log_msg}")
                else:
                    # Update the parameter
                    params['table'] = target_table_name
                    # Save the settings
                    settings.save()
                    logging.info(f"[CHANGED] {log_msg}")
            else:
                logging.debug(f"Skipping {dataset.name}: Table name already matches target format.")
        
        else:
            # Optional: Log skipped non-postgres datasets
            logging.debug(f"Skipping {dataset.name}: Type is '{settings.type}'")
            pass

    logging.info("Process complete.")

# --- Execution ---
if __name__ == "__main__":
    # Set to False to apply changes
    DRY_RUN_MODE = True
    process_postgres_datasets(dry_run=DRY_RUN_MODE)